# 02: Camada Silver: união harmonizada das 3 edições

**TC3 · PosTech FIAP Data Analytics**
Responsável: Caio Bosnic

---

**Entrada:** `bronze_dw_state_data_{2023_2024, 2024_2025, 2025_2026}`: Parquet, tudo string, particionado por ano.
**Saída:** `silver_dw_fat_respondente`: Parquet tipado, particionado por `ano_pesquisa`.
**Grão:** 1 linha por respondente por edição. **14.002 linhas.**

A Silver é o SOT: é aqui que as três pesquisas viram uma coisa só. As regras
todas vivem em `glue/silver/config_silver.py`, este notebook só orquestra e
mostra as evidências.

**Antes de rodar, ler:** [`docs/VALIDACAO_BRONZE.md`](../docs/VALIDACAO_BRONZE.md)


## O que esta camada resolve

Cinco coisas que a Bronze, sendo SOR, não pode fazer:

| # | Problema na Bronze | Tratamento aqui |
|---|---|---|
| 1 | Nenhuma edição tem coluna de ano | injeta `ano_pesquisa` a partir da origem do arquivo |
| 2 | A chave muda de nome (`id` → `token_user`) | cria `sk_respondente` = `sha2(ano \|\| id_origem)` |
| 3 | `uf` significa **moradia** em 2023/2024 e **nascimento** em 2025 | ignora `uf`; deriva a sigla de `estado` |
| 4 | 6 colunas vêm `TRUE`/`FALSE` só em 2024, `0`/`1` nas outras | mapa booleano único |
| 5 | Categorias que nasceram ou sumiram entre edições | marca `serie_comparavel = false` |

Os itens 3 e 4 são silenciosos: não quebram o job, só produzem número errado.
Foi a validação da Bronze que os encontrou.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join("..", "glue", "silver"))

# Para rodar LOCAL, aponte para a pasta com os Parquet da Bronze.
# No Glue, não defina nada: o config usa o caminho do S3.
# os.environ["TC3_PATH_BRONZE"] = "../data/bronze"
# os.environ["TC3_PATH_SILVER"] = "../data/silver"

import config_silver as cfg
from job_silver import (
    adicionar_surrogate_key,
    construir_silver,
    harmonizar,
    ler_bronze,
    marcar_comparabilidade,
    tipar,
    validar_silver,
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("tc3-silver")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 1. O contrato, em um lugar só

Trocar uma regra significa mexer em `config_silver.py` e em nenhum outro arquivo.

In [ ]:
for ano, meta in cfg.EDICOES.items():
    print(f"{ano} → {meta['edicao']:10s} | tabela={meta['tabela_bronze']:32s} "
          f"| chave={meta['col_chave']:11s} | {meta['linhas_esperadas']:>5} x {meta['colunas_esperadas']}")

print(f"\nColunas canônicas na Silver: {len(cfg.DE_PARA_COLUNAS)}")
print(f"Booleanas normalizadas:      {len(cfg.COLUNAS_BOOLEANAS)}")
print(f"Correções de origem:         {sum(len(v) for v in cfg.CORRECOES_ORIGEM.values())}")

## 2. Leitura da Bronze

`ler_bronze` conta as linhas e compara com o contrato. Se a Bronze mudar de
shape, o job para aqui em vez de gravar uma Silver silenciosamente errada.

In [ ]:
brutos = {ano: ler_bronze(spark, ano) for ano in sorted(cfg.EDICOES)}

for ano, df in brutos.items():
    print(f"{ano}: {df.count():>5} linhas x {len(df.columns):>3} colunas")

### Evidência do achado mais perigoso: `uf` troca de significado

Se a Silver usasse `uf` direto, a análise regional de 2025 seria sobre onde as
pessoas **nasceram**, não onde **moram**, e nada no pipeline acusaria erro.

In [ ]:
sigla = lambda c: F.regexp_extract(F.col(c), r"\(([A-Z]{2})\)", 1)

print("     bate com estado (moradia)   bate com estado_origem")
for ano, df in brutos.items():
    linha = f"{ano}  "
    for alvo in ("estado", "estado_origem"):
        if alvo not in df.columns or "uf" not in df.columns:
            linha += f"{'--':>22s}"
            continue
        pct = df.filter(F.col("uf") == sigla(alvo)).count() / df.count()
        linha += f"{pct:>21.1%} "
    print(linha)

### Evidência do segundo achado: `TRUE`/`FALSE` só em 2024

In [ ]:
for col in ["gestor", "vive_brasil", "empresa_possui_datalake"]:
    print(f"{col}:")
    for ano, df in brutos.items():
        if col in df.columns:
            dom = sorted(r[0] for r in df.select(col).distinct().collect() if r[0] is not None)
            print(f"   {ano}: {dom}")

## 3. Harmonizar, tipar e chavear

Uma edição de cada vez, cada uma saindo no schema canônico. Depois `unionByName`.
Casar por nome, e não por posição, deixa o job imune a uma mudança de ordem no
de-para.

In [ ]:
silver = construir_silver(spark).cache()

print(f"Linhas: {silver.count()}   Colunas: {len(silver.columns)}")
silver.printSchema()

## 4. Validação da saída

O job falha em vez de gravar Silver errada. Estas são as mesmas asserções que
`dev/tests/test_silver_local.py` roda sem precisar de AWS.

In [ ]:
validar_silver(silver)

print("\nLinhas por edição:")
(silver.groupBy("ano_pesquisa", "edicao")
       .agg(F.count("*").alias("respondentes"))
       .orderBy("ano_pesquisa")
       .show(truncate=False))

In [ ]:
# A injeção do ano_pesquisa conferida contra a data de envio real.
# 2023-2024 não coleta data, por isso a contraprova só existe em duas edições.
(silver.filter(F.col("data_envio").isNotNull())
       .groupBy("ano_pesquisa")
       .agg(F.min("data_envio").alias("primeiro_envio"),
            F.max("data_envio").alias("ultimo_envio"),
            F.count("*").alias("n"))
       .orderBy("ano_pesquisa")
       .show(truncate=False))

In [ ]:
# Booleanos: se a normalização falhasse, uma edição inteira sumiria da contagem.
(silver.groupBy("ano_pesquisa")
       .agg(F.sum(F.col("eh_gestor").cast("int")).alias("gestores"),
            F.sum(F.col("empresa_possui_datalake").cast("int")).alias("com_datalake"),
            F.sum(F.col("houve_layoff").cast("int")).alias("houve_layoff"))
       .orderBy("ano_pesquisa")
       .show())

### Comparabilidade da série

Categoria que nasceu em 2025 (ex.: senioridade `Especialista/Staff+`) parece
crescimento quando é opção nova de questionário. Quem for montar gráfico
temporal filtra `serie_comparavel = true`.

In [ ]:
(silver.groupBy("ano_pesquisa", "serie_comparavel")
       .agg(F.count("*").alias("n"))
       .orderBy("ano_pesquisa", "serie_comparavel")
       .show())

(silver.filter(~F.col("serie_comparavel"))
       .groupBy("ano_pesquisa", "nivel_senioridade", "cargo_atual")
       .agg(F.count("*").alias("n"))
       .orderBy(F.desc("n"))
       .show(10, truncate=45))

## 5. Múltipla escolha, as ~320 binárias viram 17 grupos

Cada pergunta de múltipla escolha da pesquisa vira uma coluna por opção. São
~320 das ~400 colunas. A Silver consolida cada grupo em duas colunas:

```
linguagens        "python, r, sql"
qtd_linguagens    3
```

Os grupos foram derivados em duas passadas, classificação semântica para os
seis blocos de tecnologia, máscara de resposta para o resto. A lista completa
está em `glue/silver/grupos_multipla_escolha.py`.

In [ ]:
for grupo, por_ano in sorted(cfg.GRUPOS_MULTIPLA_ESCOLHA.items()):
    tamanhos = " / ".join(str(len(por_ano.get(a, []))) for a in (2023, 2024, 2025))
    print(f"{grupo:32s} {tamanhos:>12s} opções")

In [ ]:
# Conferência: a lista consolidada tem exatamente as opções marcadas na origem?
(silver.filter((F.col("ano_pesquisa") == 2025) & (F.col("qtd_linguagens") >= 3))
       .select("cargo_atual", "linguagens", "qtd_linguagens", "clouds", "qtd_clouds")
       .show(5, truncate=48))

### ⚠️ NULL não é zero

Muitos blocos só aparecem para parte da base: `atividades_cientista_dados`
tem 2.010 respondentes de 14.002. Quem **não viu** a pergunta fica `NULL` nas
duas colunas; quem viu e não marcou nada fica `''` e `0`.

Calcular percentual sobre 14.002 quando só 2.010 viram a pergunta derruba o
número pela metade. **Sempre filtrar `IS NOT NULL` antes de dividir.**

In [ ]:
(silver.select(
    F.count("*").alias("total"),
    F.count("atividades_cientista_dados").alias("viram_o_bloco"),
    F.count(F.when(F.col("atividades_cientista_dados") == "", 1)).alias("viram_e_nao_marcaram"),
).show())

In [ ]:
# Adoção de Python por ano, o padrão de consulta que a Gold vai usar
(silver.filter(F.col("linguagens").isNotNull())          # <- o filtro que importa
       .groupBy("ano_pesquisa")
       .agg(F.count("*").alias("respondentes"),
            F.count(F.when(F.col("linguagens").contains("python"), 1)).alias("usa_python"))
       .withColumn("pct", F.round(100 * F.col("usa_python") / F.col("respondentes"), 1))
       .orderBy("ano_pesquisa")
       .show())

## 6. Gravação e catalogação

Particionado por `ano_pesquisa` para o Athena fazer partition pruning, a
maioria das perguntas de negócio filtra ou agrupa por ano.

In [ ]:
(silver.write
       .mode("overwrite")
       .partitionBy(cfg.PARTICAO_SILVER)
       .parquet(cfg.PATH_SILVER))

print(f"Gravado em {cfg.PATH_SILVER}")

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {cfg.DATABASE_GLUE}
""")

# No Athena, depois do crawler:
#   MSCK REPAIR TABLE state_of_data.silver_dw_fat_respondente;
print(f"Catalogar como {cfg.DATABASE_GLUE}.{cfg.TABELA_SILVER}")

## 7. Próximo passo

`04_gold_modelagem.ipynb`, fato e dimensões a partir desta Silver.

Combinado do grupo: cada um constrói a tabela Gold que alimenta as **suas**
perguntas de negócio, para não haver dois donos na mesma camada.

In [ ]:
spark.stop()